# Gold Transformation Pipeline

Lakeflow Declarative Pipeline stage that builds analytical tables in the **serve** schema from cleaned **refined** tables.

Produces **sixteen** gold tables: dimensional summaries, dashboard-ready KPI/trend datasets, material procurement trends, MES analytics (production + machine downtime), and **forecasting feature tables**.

**Prerequisite:** `ldp_silver_transformations` must complete successfully first.

**Workflow usage:** run as the second pipeline task, depending on the silver stage.

**Forecasting downstream:** `serve.company_forecast_features_monthly` and `serve.forecast_features_monthly` feed Prophet training notebooks.


## Configuration


In [ ]:
CATALOG = "jm_databricks_learning_ws"
REFINED_SCHEMA = "refined"
SERVE_SCHEMA = "serve"

REFINED = f"{CATALOG}.{REFINED_SCHEMA}"
SERVE = f"{CATALOG}.{SERVE_SCHEMA}"


## Imports


In [ ]:
from pyspark import pipelines as dp
from pyspark.sql.functions import (
    coalesce,
    col,
    count,
    countDistinct,
    date_format,
    explode,
    expr,
    greatest,
    lit,
    max as spark_max,
    min as spark_min,
    row_number,
    sequence,
    sum as spark_sum,
    trunc,
    unix_timestamp,
)
from pyspark.sql.window import Window

TOP_SELLING_MATERIALS_LIMIT = 10


## Helpers


In [ ]:
def _full_month_spine(bounds_df, min_col="min_month", max_col="max_month"):
    """Build a continuous month spine from inclusive min/max month columns."""
    return bounds_df.filter(
        col(min_col).isNotNull() & col(max_col).isNotNull()
    ).select(
        explode(sequence(col(min_col), col(max_col), expr("interval 1 month"))).alias(
            "month_start_date"
        )
    )

## `serve.supplier_summary`

In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.supplier_summary",
    comment="Purchase order volume and quantity by supplier",
    table_properties={"quality": "gold", "domain": "procurement"},
)
def serve_supplier_summary():
    purchase_orders = spark.read.table(f"{REFINED}.purchase_orders")
    suppliers = spark.read.table(f"{REFINED}.suppliers")

    return (
        purchase_orders.join(suppliers, on="supplier_id", how="inner")
        .groupBy(
            col("supplier_id"),
            col("supplier_name"),
            col("country"),
        )
        .agg(
            count(lit(1)).alias("total_purchase_orders"),
            spark_sum("quantity").alias("total_quantity"),
        )
    )


## `serve.customer_summary`


In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.customer_summary",
    comment="Sales order volume and quantity by customer",
    table_properties={"quality": "gold", "domain": "sales"},
)
def serve_customer_summary():
    sales_orders = spark.read.table(f"{REFINED}.sales_orders")
    customers = spark.read.table(f"{REFINED}.customers")

    return (
        sales_orders.join(customers, on="customer_id", how="inner")
        .groupBy(
            col("customer_id"),
            col("customer_name"),
            col("country"),
        )
        .agg(
            count(lit(1)).alias("total_sales_orders"),
            spark_sum("quantity").alias("total_quantity"),
        )
    )


## `serve.inventory_summary`


In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.inventory_summary",
    comment="Current stock and material coverage by warehouse",
    table_properties={"quality": "gold", "domain": "inventory"},
)
def serve_inventory_summary():
    inventory = spark.read.table(f"{REFINED}.inventory")

    latest_snapshot = inventory.agg(spark_max("snapshot_date").alias("latest_snapshot_date"))
    current_inventory = inventory.join(
        latest_snapshot,
        inventory.snapshot_date == latest_snapshot.latest_snapshot_date,
        how="inner",
    ).drop("latest_snapshot_date")

    warehouses = spark.read.table(f"{REFINED}.warehouses")

    return (
        current_inventory.join(warehouses, on="warehouse_id", how="inner")
        .groupBy(
            col("warehouse_id"),
            col("warehouse_name"),
            col("plant_id"),
        )
        .agg(
            spark_sum("quantity").alias("current_stock"),
            countDistinct("material_id").alias("total_materials"),
        )
    )


## `serve.material_summary`


In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.material_summary",
    comment="Purchased, sold, and on-hand quantities by material",
    table_properties={"quality": "gold", "domain": "inventory"},
)
def serve_material_summary():
    purchase_orders = spark.read.table(f"{REFINED}.purchase_orders")
    sales_orders = spark.read.table(f"{REFINED}.sales_orders")
    inventory = spark.read.table(f"{REFINED}.inventory")
    materials = spark.read.table(f"{REFINED}.materials")

    purchased = purchase_orders.groupBy("material_id").agg(
        spark_sum("quantity").alias("purchased_quantity"),
    )

    sold = sales_orders.groupBy("material_id").agg(
        spark_sum("quantity").alias("sold_quantity"),
    )

    latest_snapshot = inventory.agg(spark_max("snapshot_date").alias("latest_snapshot_date"))
    current_inventory = (
        inventory.join(
            latest_snapshot,
            inventory.snapshot_date == latest_snapshot.latest_snapshot_date,
            how="inner",
        )
        .groupBy("material_id")
        .agg(spark_sum("quantity").alias("current_inventory"))
    )

    return (
        materials.select("material_id", "material_name", "material_type")
        .join(purchased, on="material_id", how="left")
        .join(sold, on="material_id", how="left")
        .join(current_inventory, on="material_id", how="left")
        .select(
            "material_id",
            "material_name",
            "material_type",
            coalesce(col("purchased_quantity"), lit(0)).alias("purchased_quantity"),
            coalesce(col("sold_quantity"), lit(0)).alias("sold_quantity"),
            coalesce(col("current_inventory"), lit(0)).alias("current_inventory"),
        )
    )


## `serve.business_kpi_summary`

Single-row headline KPIs for dashboard counter widgets.

In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.business_kpi_summary",
    comment="Headline business KPIs for dashboard counter widgets",
    table_properties={"quality": "gold", "domain": "executive"},
)
def serve_business_kpi_summary():
    purchase_orders = spark.read.table(f"{REFINED}.purchase_orders")
    sales_orders = spark.read.table(f"{REFINED}.sales_orders")
    inventory = spark.read.table(f"{REFINED}.inventory")

    active_purchase_orders = purchase_orders.filter(col("status") != "CANCELLED")
    active_sales_orders = sales_orders.filter(col("status") != "CANCELLED")

    latest_snapshot = inventory.agg(spark_max("snapshot_date").alias("inventory_snapshot_date"))
    current_inventory = inventory.join(
        latest_snapshot,
        inventory.snapshot_date == latest_snapshot.inventory_snapshot_date,
        how="inner",
    )

    purchase_stats = active_purchase_orders.agg(
        count(lit(1)).alias("total_purchase_orders"),
        countDistinct("supplier_id").alias("active_suppliers"),
    )
    sales_stats = active_sales_orders.agg(
        count(lit(1)).alias("total_sales_orders"),
        countDistinct("customer_id").alias("active_customers"),
    )
    inventory_stats = current_inventory.agg(
        coalesce(spark_sum("quantity"), lit(0)).alias("inventory_on_hand"),
    )

    return (
        purchase_stats.crossJoin(sales_stats)
        .crossJoin(inventory_stats)
        .crossJoin(latest_snapshot)
    )

## `serve.sales_trend_monthly`

Monthly sales order volume across the full order history (line chart).

In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.sales_trend_monthly",
    comment="Monthly sales order count and quantity for dashboard trend charts",
    table_properties={"quality": "gold", "domain": "sales"},
)
def serve_sales_trend_monthly():
    sales_orders = (
        spark.read.table(f"{REFINED}.sales_orders")
        .filter(col("status") != "CANCELLED")
        .filter(col("order_date").isNotNull())
    )

    bounds = sales_orders.agg(
        spark_min(trunc("order_date", "month")).alias("min_month"),
        spark_max(trunc("order_date", "month")).alias("max_month"),
    )
    months = _full_month_spine(bounds)

    monthly_totals = (
        sales_orders.withColumn("month_start_date", trunc("order_date", "month"))
        .groupBy("month_start_date")
        .agg(
            count(lit(1)).alias("order_count"),
            spark_sum("quantity").alias("total_quantity"),
        )
    )

    return (
        months.join(monthly_totals, on="month_start_date", how="left")
        .select(
            col("month_start_date"),
            date_format("month_start_date", "yyyy-MM").alias("year_month"),
            coalesce(col("order_count"), lit(0)).alias("order_count"),
            coalesce(col("total_quantity"), lit(0)).alias("total_quantity"),
        )
        .orderBy("month_start_date")
    )

## `serve.purchase_trend_monthly`

Monthly purchase order volume across the full order history (line chart).

In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.purchase_trend_monthly",
    comment="Monthly purchase order count and quantity for dashboard trend charts",
    table_properties={"quality": "gold", "domain": "procurement"},
)
def serve_purchase_trend_monthly():
    purchase_orders = (
        spark.read.table(f"{REFINED}.purchase_orders")
        .filter(col("status") != "CANCELLED")
        .filter(col("order_date").isNotNull())
    )

    bounds = purchase_orders.agg(
        spark_min(trunc("order_date", "month")).alias("min_month"),
        spark_max(trunc("order_date", "month")).alias("max_month"),
    )
    months = _full_month_spine(bounds)

    monthly_totals = (
        purchase_orders.withColumn("month_start_date", trunc("order_date", "month"))
        .groupBy("month_start_date")
        .agg(
            count(lit(1)).alias("order_count"),
            spark_sum("quantity").alias("total_quantity"),
        )
    )

    return (
        months.join(monthly_totals, on="month_start_date", how="left")
        .select(
            col("month_start_date"),
            date_format("month_start_date", "yyyy-MM").alias("year_month"),
            coalesce(col("order_count"), lit(0)).alias("order_count"),
            coalesce(col("total_quantity"), lit(0)).alias("total_quantity"),
        )
        .orderBy("month_start_date")
    )

## `serve.top_selling_materials`

Top finished goods by sold quantity (bar chart).

In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.top_selling_materials",
    comment="Top finished goods ranked by sold quantity for dashboard bar charts",
    table_properties={"quality": "gold", "domain": "sales"},
)
def serve_top_selling_materials():
    sales_orders = spark.read.table(f"{REFINED}.sales_orders").filter(
        col("status") != "CANCELLED"
    )
    materials = spark.read.table(f"{REFINED}.materials").filter(
        col("material_type") == "FINISHED_GOOD"
    )

    sold_by_material = sales_orders.groupBy("material_id").agg(
        spark_sum("quantity").alias("sold_quantity"),
    )

    ranked = (
        sold_by_material.join(materials, on="material_id", how="inner")
        .withColumn(
            "sales_rank",
            row_number().over(Window.orderBy(col("sold_quantity").desc(), col("material_id"))),
        )
        .filter(col("sales_rank") <= TOP_SELLING_MATERIALS_LIMIT)
    )

    return ranked.select(
        "sales_rank",
        "material_id",
        "material_name",
        "material_type",
        "sold_quantity",
    ).orderBy("sales_rank")

## `serve.material_procurement_trend_monthly`

Monthly material-level procurement, sales, and production consumption metrics for the material procurement dashboard.

In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.material_procurement_trend_monthly",
    comment="Monthly PO/SO and production quantities by material for procurement dashboards",
    table_properties={"quality": "gold", "domain": "procurement"},
)
def serve_material_procurement_trend_monthly():
    purchase_orders = (
        spark.read.table(f"{REFINED}.purchase_orders")
        .filter(col("status") != "CANCELLED")
        .filter(col("order_date").isNotNull())
    )
    sales_orders = (
        spark.read.table(f"{REFINED}.sales_orders")
        .filter(col("status") != "CANCELLED")
        .filter(col("order_date").isNotNull())
    )
    production_output = spark.read.table(f"{REFINED}.production_output")
    production_orders = (
        spark.read.table(f"{REFINED}.production_orders")
        .filter(col("status") != "CANCELLED")
    )
    materials = spark.read.table(f"{REFINED}.materials")

    po_monthly = (
        purchase_orders.withColumn("month_start_date", trunc("order_date", "month"))
        .groupBy("material_id", "month_start_date")
        .agg(
            count(lit(1)).alias("purchase_order_count"),
            spark_sum("quantity").alias("purchase_quantity"),
        )
    )

    so_monthly = (
        sales_orders.withColumn("month_start_date", trunc("order_date", "month"))
        .groupBy("material_id", "month_start_date")
        .agg(
            count(lit(1)).alias("sales_order_count"),
            spark_sum("quantity").alias("sales_quantity"),
        )
    )

    production_with_month = (
        production_output.join(
            production_orders.select("production_order_id", "start_date", "end_date"),
            on="production_order_id",
            how="inner",
        )
        .withColumn(
            "month_start_date",
            trunc(coalesce(col("end_date"), col("start_date")), "month"),
        )
        .filter(col("month_start_date").isNotNull())
    )

    input_monthly = production_with_month.groupBy(
        col("input_material_id").alias("material_id"),
        "month_start_date",
    ).agg(spark_sum("input_quantity").alias("production_input_quantity"))

    output_monthly = production_with_month.groupBy(
        col("output_material_id").alias("material_id"),
        "month_start_date",
    ).agg(spark_sum("output_quantity").alias("production_output_quantity"))

    material_months = (
        po_monthly.select("material_id", "month_start_date")
        .union(so_monthly.select("material_id", "month_start_date"))
        .union(input_monthly.select("material_id", "month_start_date"))
        .union(output_monthly.select("material_id", "month_start_date"))
        .distinct()
    )

    return (
        material_months.join(
            materials.select("material_id", "material_name", "material_type"),
            on="material_id",
            how="inner",
        )
        .join(po_monthly, on=["material_id", "month_start_date"], how="left")
        .join(so_monthly, on=["material_id", "month_start_date"], how="left")
        .join(input_monthly, on=["material_id", "month_start_date"], how="left")
        .join(output_monthly, on=["material_id", "month_start_date"], how="left")
        .select(
            "material_id",
            "material_name",
            "material_type",
            "month_start_date",
            date_format("month_start_date", "yyyy-MM").alias("year_month"),
            coalesce(col("purchase_order_count"), lit(0)).alias("purchase_order_count"),
            coalesce(col("purchase_quantity"), lit(0)).alias("purchase_quantity"),
            coalesce(col("sales_order_count"), lit(0)).alias("sales_order_count"),
            coalesce(col("sales_quantity"), lit(0)).alias("sales_quantity"),
            coalesce(col("production_input_quantity"), lit(0)).alias("production_input_quantity"),
            coalesce(col("production_output_quantity"), lit(0)).alias("production_output_quantity"),
        )
        .orderBy("material_id", "month_start_date")
    )

## `serve.production_order_summary`

Production order counts and quantities by plant and status.

In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.production_order_summary",
    comment="Production order volume and quantities by plant and status",
    table_properties={"quality": "gold", "domain": "mes"},
)
def serve_production_order_summary():
    production_orders = spark.read.table(f"{REFINED}.production_orders")
    plants = spark.read.table(f"{REFINED}.plants")

    return (
        production_orders.join(plants, on="plant_id", how="inner")
        .groupBy(
            col("plant_id"),
            col("plant_name"),
            col("status"),
        )
        .agg(
            count(lit(1)).alias("order_count"),
            spark_sum("planned_quantity").alias("planned_quantity"),
            coalesce(spark_sum("actual_quantity"), lit(0)).alias("actual_quantity"),
        )
    )

## `serve.production_trend_monthly`

Monthly production order volume across the full production history.

In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.production_trend_monthly",
    comment="Monthly production order count and planned/actual quantities",
    table_properties={"quality": "gold", "domain": "mes"},
)
def serve_production_trend_monthly():
    production_orders = (
        spark.read.table(f"{REFINED}.production_orders")
        .filter(col("status") != "CANCELLED")
        .withColumn(
            "production_month",
            trunc(coalesce(col("end_date"), col("start_date")), "month"),
        )
        .filter(col("production_month").isNotNull())
    )

    bounds = production_orders.agg(
        spark_min("production_month").alias("min_month"),
        spark_max("production_month").alias("max_month"),
    )
    months = _full_month_spine(bounds)

    monthly_totals = production_orders.groupBy(
        col("production_month").alias("month_start_date")
    ).agg(
        count(lit(1)).alias("order_count"),
        spark_sum("planned_quantity").alias("planned_quantity"),
        coalesce(spark_sum("actual_quantity"), lit(0)).alias("actual_quantity"),
    )

    return (
        months.join(monthly_totals, on="month_start_date", how="left")
        .select(
            col("month_start_date"),
            date_format("month_start_date", "yyyy-MM").alias("year_month"),
            coalesce(col("order_count"), lit(0)).alias("order_count"),
            coalesce(col("planned_quantity"), lit(0)).alias("planned_quantity"),
            coalesce(col("actual_quantity"), lit(0)).alias("actual_quantity"),
        )
        .orderBy("month_start_date")
    )

## `serve.machine_downtime_summary`

Machine downtime event counts and hours by plant, machine, and reason.

In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.machine_downtime_summary",
    comment="Downtime event counts and total hours by plant, machine, and reason",
    table_properties={"quality": "gold", "domain": "mes"},
)
def serve_machine_downtime_summary():
    machine_downtime = spark.read.table(f"{REFINED}.machine_downtime")
    plants = spark.read.table(f"{REFINED}.plants")

    enriched = machine_downtime.withColumn(
        "downtime_hours",
        (unix_timestamp("end_time") - unix_timestamp("start_time")) / lit(3600.0),
    ).filter(col("downtime_hours") > 0)

    return (
        enriched.join(plants, on="plant_id", how="inner")
        .groupBy(
            col("plant_id"),
            col("plant_name"),
            col("machine_name"),
            col("reason"),
        )
        .agg(
            count(lit(1)).alias("downtime_event_count"),
            spark_sum("downtime_hours").alias("total_downtime_hours"),
        )
    )

## `serve.machine_downtime_monthly`

Monthly downtime trends by plant and reason across the full event history.

In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.machine_downtime_monthly",
    comment="Monthly downtime event counts and hours by plant and reason",
    table_properties={"quality": "gold", "domain": "mes"},
)
def serve_machine_downtime_monthly():
    machine_downtime = spark.read.table(f"{REFINED}.machine_downtime")
    plants = spark.read.table(f"{REFINED}.plants")

    enriched = (
        machine_downtime.withColumn(
            "downtime_hours",
            (unix_timestamp("end_time") - unix_timestamp("start_time")) / lit(3600.0),
        )
        .withColumn("month_start_date", trunc("start_time", "month"))
        .filter(col("downtime_hours") > 0)
        .filter(col("month_start_date").isNotNull())
    )

    return (
        enriched.join(plants, on="plant_id", how="inner")
        .groupBy("plant_id", "plant_name", "month_start_date", "reason")
        .agg(
            count(lit(1)).alias("downtime_event_count"),
            spark_sum("downtime_hours").alias("total_downtime_hours"),
        )
        .select(
            "plant_id",
            "plant_name",
            "month_start_date",
            date_format("month_start_date", "yyyy-MM").alias("year_month"),
            "reason",
            "downtime_event_count",
            "total_downtime_hours",
        )
        .orderBy("plant_id", "month_start_date", "reason")
    )

## `serve.inventory_monthly`

Month-end inventory on hand per material (sum across warehouses on the last snapshot day in each month). Used as an exogenous signal for sales forecasting.

In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.inventory_monthly",
    comment="Month-end inventory on hand per material for forecasting features",
    table_properties={"quality": "gold", "domain": "inventory"},
)
def serve_inventory_monthly():
    inventory = spark.read.table(f"{REFINED}.inventory")

    daily = (
        inventory.withColumn("month_start_date", trunc("snapshot_date", "month"))
        .groupBy("material_id", "snapshot_date", "month_start_date")
        .agg(spark_sum("quantity").alias("quantity"))
    )

    month_end_dates = daily.groupBy("material_id", "month_start_date").agg(
        spark_max("snapshot_date").alias("snapshot_date")
    )

    return (
        daily.join(
            month_end_dates,
            on=["material_id", "month_start_date", "snapshot_date"],
            how="inner",
        )
        .select(
            "material_id",
            "month_start_date",
            date_format("month_start_date", "yyyy-MM").alias("year_month"),
            col("quantity").alias("inventory_on_hand"),
        )
        .orderBy("material_id", "month_start_date")
    )

## `serve.company_forecast_features_monthly`

Company-level monthly feature matrix for forecasting: sales, procurement, production, inventory, and downtime.

In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.company_forecast_features_monthly",
    comment="Company-level monthly features for sales forecasting models",
    table_properties={"quality": "gold", "domain": "forecasting"},
)
def serve_company_forecast_features_monthly():
    sales = spark.read.table(f"{SERVE}.sales_trend_monthly").select(
        "month_start_date",
        col("order_count").alias("sales_order_count"),
        col("total_quantity").alias("sales_quantity"),
    )

    procurement = spark.read.table(f"{SERVE}.material_procurement_trend_monthly")

    purchase = procurement.groupBy("month_start_date").agg(
        spark_sum("purchase_quantity").alias("purchase_quantity")
    )

    production = (
        procurement.filter(col("material_type") == "FINISHED_GOOD")
        .groupBy("month_start_date")
        .agg(spark_sum("production_output_quantity").alias("production_output_quantity"))
    )

    inventory = spark.read.table(f"{SERVE}.inventory_monthly").groupBy("month_start_date").agg(
        spark_sum("inventory_on_hand").alias("inventory_on_hand")
    )

    downtime = (
        spark.read.table(f"{SERVE}.machine_downtime_monthly")
        .groupBy("month_start_date")
        .agg(spark_sum("total_downtime_hours").alias("downtime_hours"))
    )

    return (
        sales.join(purchase, on="month_start_date", how="left")
        .join(production, on="month_start_date", how="left")
        .join(inventory, on="month_start_date", how="left")
        .join(downtime, on="month_start_date", how="left")
        .select(
            "month_start_date",
            date_format("month_start_date", "yyyy-MM").alias("year_month"),
            "sales_order_count",
            "sales_quantity",
            coalesce(col("purchase_quantity"), lit(0)).alias("purchase_quantity"),
            coalesce(col("production_output_quantity"), lit(0)).alias(
                "production_output_quantity"
            ),
            coalesce(col("inventory_on_hand"), lit(0)).alias("inventory_on_hand"),
            coalesce(col("downtime_hours"), lit(0.0)).alias("downtime_hours"),
        )
        .orderBy("month_start_date")
    )

## `serve.forecast_features_monthly`

Dense month spine for top-selling finished goods with procurement, inventory, and downtime signals. Used for material-level forecasting models.

In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.forecast_features_monthly",
    comment="Top-SKU monthly feature matrix for material-level sales forecasting",
    table_properties={"quality": "gold", "domain": "forecasting"},
)
def serve_forecast_features_monthly():
    top_mats = spark.read.table(f"{SERVE}.top_selling_materials").select(
        "material_id", "material_name"
    )
    procurement = (
        spark.read.table(f"{SERVE}.material_procurement_trend_monthly")
        .select(
            "material_id",
            "month_start_date",
            "sales_order_count",
            "sales_quantity",
            "purchase_order_count",
            "purchase_quantity",
            "production_input_quantity",
            "production_output_quantity",
        )
    )
    inventory_monthly = spark.read.table(f"{SERVE}.inventory_monthly").select(
        "material_id", "month_start_date", "inventory_on_hand"
    )

    bounds = procurement.agg(
        spark_min("month_start_date").alias("min_month"),
        spark_max("month_start_date").alias("max_month"),
    )
    months = _full_month_spine(bounds)
    spine = top_mats.crossJoin(months)

    downtime_total = (
        spark.read.table(f"{SERVE}.machine_downtime_monthly")
        .groupBy("month_start_date")
        .agg(spark_sum("total_downtime_hours").alias("plant_downtime_hours"))
    )

    top_n = top_mats.agg(count(lit(1)).alias("top_n"))

    return (
        spine.join(procurement, on=["material_id", "month_start_date"], how="left")
        .join(inventory_monthly, on=["material_id", "month_start_date"], how="left")
        .join(downtime_total, on="month_start_date", how="left")
        .crossJoin(top_n)
        .select(
            col("material_id"),
            col("material_name"),
            col("month_start_date"),
            date_format("month_start_date", "yyyy-MM").alias("year_month"),
            coalesce(col("sales_order_count"), lit(0)).alias("sales_order_count"),
            coalesce(col("sales_quantity"), lit(0)).alias("sales_quantity"),
            coalesce(col("purchase_order_count"), lit(0)).alias("purchase_order_count"),
            coalesce(col("purchase_quantity"), lit(0)).alias("purchase_quantity"),
            coalesce(col("production_input_quantity"), lit(0)).alias(
                "production_input_quantity"
            ),
            coalesce(col("production_output_quantity"), lit(0)).alias(
                "production_output_quantity"
            ),
            coalesce(col("inventory_on_hand"), lit(0)).alias("inventory_on_hand"),
            (
                coalesce(col("plant_downtime_hours"), lit(0.0))
                / greatest(col("top_n"), lit(1))
            ).alias("downtime_hours"),
        )
        .orderBy("material_id", "month_start_date")
    )